> Текущий этап v0.4: [04_pathpack_v04.ipynb](04_pathpack_v04.ipynb). Этот блокнот сохранён для воспроизведения этапа v0.3.

# CodeGraph-RAG v0.3 — единый блокнот PathPack / SWE-bench

Здесь объединены подготовка SWE-bench из v0.2 и инструменты v0.3: четыре метода (`bm25`, `graph-expand`, `pathpack`, `pathpack-v3`), отдельная оценка функций, аудит бюджета и парный bootstrap. Пакет установлен из корня текущего проекта. Запуск: `source .venv/bin/activate`, затем `jupyter notebook notebooks/03_swebench_pathpack.ipynb`. После обновления исходников перезапустите kernel.

Выборка: 80 задач, 20 dev / 60 test, seed 42. Существующие манифесты и результаты сохраняются. Новый прогон пишет в `output/benchmark/v03/`, по умолчанию только dev. Ранее просмотренные 60 test-задач подходят для ретроспективного сравнения; для подтверждающего исследования нужен новый заранее зафиксированный holdout.

Budget — `ceil(UTF-8 bytes / 4)`, proxy токенов. Нужен один настоящий токенизатор перед выводами о лимите контекста Gemma. Gold patch используется только после retrieval, в scoring. Docker для этой оценки не нужен.


In [ ]:
from pathlib import Path
import sys, os, json, re, time, subprocess, math
import pandas as pd
ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').exists(): ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), 'Откройте блокнот из корня CodeGraph-RAG'
sys.path.insert(0, str(ROOT / 'src'))
os.environ.setdefault('HF_DATASETS_CACHE', str(ROOT / 'output' / 'benchmark' / 'hf_cache'))
if list((ROOT / 'output' / 'benchmark' / 'hf_cache').glob('SWE-bench___swe-bench_verified/default/0.0.0/*')): os.environ.setdefault('HF_HUB_OFFLINE', '1')
from datasets import load_dataset
from codegraph_rag.benchmark import build_manifests, read_jsonl
from codegraph_rag.parser import scan_repository
from codegraph_rag.retrieval import make_chunks, select_context
from codegraph_rag.audit import audit_predictions, paired_bootstrap
from codegraph_rag.function_gold import functions_touched
BUDGETS = (1024, 2048, 4096)
OUT = ROOT / 'output' / 'benchmark'
ISSUES = OUT / 'issues.jsonl'
GOLD = OUT / 'gold_labels.jsonl'  # открывать только в scoring-ячейках
SNAPSHOTS = ROOT / 'data' / 'swebench' / 'snapshots'
MIRRORS = ROOT / 'data' / 'swebench' / 'git_mirrors'
OUT.mkdir(parents=True, exist_ok=True)
RUN_OUT = OUT / 'v03'
RUN_OUT.mkdir(parents=True, exist_ok=True)
RUN_SPLITS = ('dev',)
EVALUATE_REVIEWED_TEST = False  # ретроспективная проверка уже просмотренных 60 test
print('Project:', ROOT, '| outputs:', RUN_OUT, '| splits:', RUN_SPLITS)


## 1. Датасет и фиксированный манифест

При первом запуске создаётся выборка 80 задач с seed 42. Существующий манифест читается без перезаписи; gold labels остаются отдельно. Все методы получают только `issues.jsonl`. Существующая таблица v0.2 и агрегаты из архива показаны ниже с указанием происхождения, без приписывания их v0.3.


In [ ]:
dataset = load_dataset('SWE-bench/SWE-bench_Verified', split='test')
print('Всего задач:', len(dataset), '| пример:', dataset[0]['instance_id'])
assert len(dataset) == 500, f'Ожидали 500 задач, получено {len(dataset)}'
if not ISSUES.exists() and not GOLD.exists():
    build_manifests(dataset, OUT, limit=80, dev_count=20, seed=42)
assert ISSUES.exists() and GOLD.exists(), 'Нужны оба файла манифеста; восстановите отсутствующий файл'
issues = read_jsonl(ISSUES)
assert len({r['instance_id'] for r in issues}) == 80, 'Требуются 80 уникальных задач'
assert sum(r['split']=='dev' for r in issues)==20 and sum(r['split']=='test' for r in issues)==60
assert all('patch' not in row for row in issues)
experiment_issues = [r for r in issues if r['split'] in RUN_SPLITS]
display(pd.DataFrame(issues).groupby('split').agg(tasks=('instance_id','count'), repos=('repo','nunique')))
legacy_results = OUT / 'main_results.csv'
if legacy_results.exists():
    print('Сохранённая таблица прежнего прогона (не v0.3):')
    display(pd.read_csv(legacy_results))


## 2. Снимки репозиториев на точном `base_commit`
Git worktree разделяет объектную базу репозитория между задачами и позволяет не клонировать одну и ту же библиотеку повторно. Ячейка возобновляемая: корректные существующие снимки пропускаются. Загрузка репозиториев может занять время и место на диске; `TASK_LIMIT` можно временно уменьшить для проверки. Перед итоговым измерением верните `None`.

In [ ]:
TASK_LIMIT = None  # например 3 для пробного запуска; None = все 80
SNAPSHOTS.mkdir(parents=True, exist_ok=True); MIRRORS.mkdir(parents=True, exist_ok=True)
def git(args, *, check=True):
    return subprocess.run(['git', *map(str,args)], text=True, capture_output=True, check=check)
def snapshot_for(row):
    repo_key = row['repo'].replace('/', '__')
    mirror = MIRRORS / (repo_key + '.git')
    snap = SNAPSHOTS / row['instance_id']
    if snap.exists():
        head = git(['-C', snap, 'rev-parse', 'HEAD']).stdout.strip()
        if head == row['base_commit']: return 'ready'
        raise RuntimeError(f'{snap} существует, но HEAD={head}; ожидается {row["base_commit"]}')
    if not mirror.exists():
        git(['clone', '--bare', '--filter=blob:none', f'https://github.com/{row["repo"]}.git', mirror])
    exists = git(['--git-dir', mirror, 'cat-file', '-e', row['base_commit'] + '^{commit}'], check=False)
    if exists.returncode: git(['--git-dir', mirror, 'fetch', '--filter=blob:none', 'origin', row['base_commit']])
    git(['--git-dir', mirror, 'worktree', 'add', '--detach', snap, row['base_commit']])
    head = git(['-C', snap, 'rev-parse', 'HEAD']).stdout.strip()
    if head != row['base_commit']: raise RuntimeError(f'Неверный commit для {row["instance_id"]}')
    return 'created'
todo = issues if TASK_LIMIT is None else issues[:TASK_LIMIT]
snapshot_status = []
for i, row in enumerate(todo, 1):
    snapshot_status.append(snapshot_for(row))
    if i % 10 == 0 or i == len(todo): print(f'{i}/{len(todo)} снимков')
print(pd.Series(snapshot_status).value_counts().to_dict())

In [ ]:
# Убедиться, что доступны все snapshots и каждый pinned commit совпадает.
snapshot_check = []
for row in issues:
    p = SNAPSHOTS / row['instance_id']
    if not p.is_dir(): continue
    head = git(['-C', p, 'rev-parse', 'HEAD']).stdout.strip()
    snapshot_check.append({'instance_id': row['instance_id'], 'expected': row['base_commit'], 'head': head, 'ok': head == row['base_commit']})
snapshot_check = pd.DataFrame(snapshot_check, columns=['instance_id','expected','head','ok'])
display(snapshot_check.groupby('ok').size().rename('snapshots'))
assert len(snapshot_check) == 80 and snapshot_check['ok'].all(), 'Подготовьте все 80 точных снимков перед экспериментом'

## 3. Графы задач выбранного split

Парсер работает с Python без исполнения кода репозиториев. Строим граф один раз на задачу выбранного split и используем одинаковые chunks для всех четырёх методов. Время AST-сканирования публикуется отдельно от latency retrieval. Ошибки парсера и размер графа сохраняются в `v03/graph_inventory.csv`.


In [ ]:
graph_cache = {}
graph_rows = []
for i, row in enumerate(experiment_issues, 1):
    repo_path = SNAPSHOTS / row['instance_id']
    t0 = time.perf_counter(); graph = scan_repository(repo_path); scan_sec = time.perf_counter() - t0
    chunks = make_chunks(repo_path, graph)
    graph_cache[row['instance_id']] = (graph, chunks)
    graph_rows.append({'instance_id': row['instance_id'], 'repo': row['repo'], 'scan_sec': scan_sec, 'chunks': len(chunks), **{k:v for k,v in graph.stats().items() if k != 'parse_errors'}, 'parse_error_files': len(graph.errors)})
    if i % 10 == 0 or i == len(experiment_issues): print(f'{i}/{len(experiment_issues)} графов')
graph_df = pd.DataFrame(graph_rows)
graph_df.to_csv(RUN_OUT / 'graph_inventory.csv', index=False)
display(graph_df[['scan_sec','chunks','nodes','edges','parse_error_files']].describe().round(2))

## 4. Четыре метода при одинаковых бюджетах

Legacy `pathpack` сохраняет настройку `redundancy_penalty` для абляций. `pathpack-v3` использует фиксированный порядок с разнообразием файлов и упаковку префикса, что даёт вложенные наборы при росте бюджета. Он может оставлять бюджет неиспользованным. Latency включает BM25 и выбор контекста, AST-сканирование измерено отдельно.


In [ ]:
METHODS = ('bm25', 'graph-expand', 'pathpack', 'pathpack-v3')
predictions = []
for i, row in enumerate(experiment_issues, 1):
    graph, chunks = graph_cache[row['instance_id']]
    for budget in BUDGETS:
        for method in METHODS:
            t0 = time.perf_counter()
            hits = select_context(chunks, graph, row['problem_statement'], method=method, budget=budget, seeds=3, hops=3)
            elapsed = time.perf_counter() - t0
            predictions.append({'instance_id': row['instance_id'], 'split': row['split'], 'method': method, 'budget': budget, 'latency_sec': elapsed, 'tokens_est': sum(h.chunk.tokens_est for h in hits), 'selected': [h.chunk.metadata() for h in hits]})
    if i % 10 == 0 or i == len(experiment_issues): print(f'{i}/{len(experiment_issues)} задач обработано')
prediction_paths = {}
for split in RUN_SPLITS:
    path = RUN_OUT / f'predictions_{split}.jsonl'
    with path.open('w', encoding='utf-8') as f:
        for prediction in predictions:
            if prediction['split'] == split:
                f.write(json.dumps(prediction, ensure_ascii=False) + '\n')
    prediction_paths[split] = path
print('Предсказаний:', len(predictions))


## 5. Scoring: изменённые строки → AST-функции

Разметка v0.3 использует именно изменённые старые строки, а не весь diff hunk с неизменённым контекстом. Для вложенных функций выбирается наименьший AST-диапазон. Вставки привязываются к соседним старым строкам; новые файлы и изменения вне функций размечаются неполно. Этот proxy отличается от прежней разметки блокнота — старые таблицы не пересчитываются молча.


In [ ]:
gold_rows = {r['instance_id']: r for r in read_jsonl(GOLD)}
source_rows = {r['instance_id']: r for r in dataset}
gold_functions = {}
for iid, (graph, _) in graph_cache.items():
    gold_functions[iid] = set(functions_touched(source_rows[iid]['patch'], SNAPSHOTS / iid, graph=graph))
scoring_gold = []
for row in gold_rows.values():
    enriched = dict(row)
    if row['instance_id'] in gold_functions:
        enriched['gold_functions'] = sorted(gold_functions[row['instance_id']])
        enriched['function_label_method'] = 'changed_base_lines_to_innermost_AST_function'
    scoring_gold.append(enriched)
SCORING_GOLD = RUN_OUT / 'gold_functions.jsonl'
with SCORING_GOLD.open('w', encoding='utf-8') as f:
    for row in scoring_gold:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')
def recall_rows(predictions):
    scored=[]
    for p in predictions:
        labels = gold_rows[p['instance_id']]['gold_python_files']
        if not labels: continue
        selected_files = {x['path'] for x in p['selected']}
        file_hit = len(selected_files & set(labels)) / len(set(labels))
        targets = gold_functions[p['instance_id']]
        selected_ids = {x['id'] for x in p['selected']}
        fn_recall = len(selected_ids & targets) / len(targets) if targets else math.nan
        scored.append({**{k:p[k] for k in ('instance_id','split','method','budget','latency_sec','tokens_est')}, **{k:p[k] for k in ('graph_bonus','hops','redundancy_penalty') if k in p}, 'file_recall':file_hit, 'function_recall':fn_recall, 'function_targets':len(targets), 'useful_functions_per_1k_est_tokens':(len(selected_ids & targets) * 1000 / p['tokens_est']) if p['tokens_est'] else 0})
    return pd.DataFrame(scored)
scores = recall_rows(predictions)
summary = scores.groupby(['split','budget','method']).agg(tasks=('instance_id','nunique'), file_recall=('file_recall','mean'), function_recall=('function_recall','mean'), latency_sec=('latency_sec','mean'), tokens_est=('tokens_est','mean'), useful_functions_per_1k=('useful_functions_per_1k_est_tokens','mean')).reset_index()
summary.to_csv(RUN_OUT / 'main_results.csv', index=False)
display(summary.round(4))
print('File-recall coverage:', len({x['instance_id'] for x in issues if gold_rows[x['instance_id']]['gold_python_files']}), 'of', len(issues), 'tasks; without gold Python files:', sum(not gold_rows[x['instance_id']]['gold_python_files'] for x in issues))
print('Function recall evaluated:', int(scores['function_recall'].notna().sum()), '| excluded:', int(scores['function_recall'].isna().sum()))

## 6. Абляции legacy PathPack на dev

Оцениваем графовый бонус, длину обхода и Jaccard-штраф сходства с уже выбранными фрагментами. Штраф ноль воспроизводит legacy PathPack. Это отдельное исследование от фиксированного ранжирования `pathpack-v3`; параметры v3 здесь не подбираются.


In [ ]:
ABLATION = [(g,h,r) for g in (0.0, 0.75, 1.5) for h in (1,2,3) for r in (0.0,0.25,0.5)]
dev_rows = [x for x in experiment_issues if x['split'] == 'dev']
assert dev_rows, 'Для абляций включите dev в RUN_SPLITS'
ablation_predictions=[]
for row in dev_rows:
    graph, chunks = graph_cache[row['instance_id']]
    for budget in BUDGETS:
        for graph_bonus, hops, redundancy_penalty in ABLATION:
            t0=time.perf_counter()
            hits=select_context(chunks, graph, row['problem_statement'], method='pathpack', budget=budget, seeds=3, hops=hops, graph_bonus=graph_bonus, redundancy_penalty=redundancy_penalty)
            ablation_predictions.append({'instance_id':row['instance_id'],'split':'dev','method':'pathpack','budget':budget,'graph_bonus':graph_bonus,'hops':hops,'redundancy_penalty':redundancy_penalty,'latency_sec':time.perf_counter()-t0,'tokens_est':sum(h.chunk.tokens_est for h in hits),'selected':[h.chunk.metadata() for h in hits]})
ablation_scores=recall_rows(ablation_predictions)
ablation_summary=ablation_scores.groupby(['budget','graph_bonus','hops','redundancy_penalty']).agg(tasks=('instance_id','nunique'),file_recall=('file_recall','mean'),function_recall=('function_recall','mean'),latency_sec=('latency_sec','mean'),tokens_est=('tokens_est','mean')).reset_index()
ablation_summary.to_csv(RUN_OUT / 'dev_ablations.csv',index=False)
display(ablation_summary[ablation_summary['budget']==2048].sort_values(['file_recall','function_recall'],ascending=False).head(15).round(4))
with (RUN_OUT / 'dev_ablation_predictions.jsonl').open('w', encoding='utf-8') as f:
    for prediction in ablation_predictions:
        f.write(json.dumps(prediction, ensure_ascii=False) + '\n')


## 7. Фиксация параметров и необязательное ретроспективное сравнение

Выбираем legacy PathPack по dev File Recall@2048; равенства разрешаем меньшим штрафом, hops и бонусом. Конфигурация сохраняется. Ячейка запускает просмотренный test только при `EVALUATE_REVIEWED_TEST = True` в настройках. Такое сравнение нельзя объявлять новым нетронутым holdout.


In [ ]:
dev2048=ablation_summary[ablation_summary['budget']==2048].sort_values(['file_recall','redundancy_penalty','hops','graph_bonus'],ascending=[False,True,True,True])
best=dev2048.iloc[0]
frozen={'graph_bonus':float(best.graph_bonus),'hops':int(best.hops),'redundancy_penalty':float(best.redundancy_penalty),'selection_rule':'max dev file recall @2048; tie: lower redundancy, fewer hops, lower graph bonus'}
print('Замороженные параметры:', frozen)
(RUN_OUT / 'frozen_config.json').write_text(json.dumps(frozen,indent=2)+'\n',encoding='utf-8')
if not EVALUATE_REVIEWED_TEST:
    print('Параметры сохранены. Ретроспективный test отключён.')
else:
    final_test=[]
    for row in [x for x in issues if x['split']=='test']:
        iid = row['instance_id']
        repo_path = SNAPSHOTS / iid
        assert git(['-C', repo_path, 'rev-parse', 'HEAD']).stdout.strip() == row['base_commit']
        if iid not in graph_cache:
            graph = scan_repository(repo_path)
            graph_cache[iid] = (graph, make_chunks(repo_path, graph))
        graph,chunks=graph_cache[iid]
        for budget in BUDGETS:
            t0=time.perf_counter()
            hits=select_context(chunks,graph,row['problem_statement'],method='pathpack',budget=budget,seeds=3,**{k:frozen[k] for k in ('graph_bonus','hops','redundancy_penalty')})
            final_test.append({'instance_id':row['instance_id'],'split':'test','method':'pathpack-dev-tuned','budget':budget,'latency_sec':time.perf_counter()-t0,'tokens_est':sum(h.chunk.tokens_est for h in hits),'selected':[h.chunk.metadata() for h in hits]})
    for row in [x for x in issues if x['split']=='test']:
        iid=row['instance_id']
        gold_functions[iid]=set(functions_touched(source_rows[iid]['patch'],SNAPSHOTS/iid,graph=graph_cache[iid][0]))
    with (RUN_OUT / 'reviewed_test_predictions.jsonl').open('w',encoding='utf-8') as f:
        for prediction in final_test: f.write(json.dumps(prediction,ensure_ascii=False)+'\n')
    final_scores=recall_rows(final_test)
    final_summary=final_scores.groupby(['budget','method']).agg(tasks=('instance_id','nunique'),file_recall=('file_recall','mean'),function_recall=('function_recall','mean'),latency_sec=('latency_sec','mean'),tokens_est=('tokens_est','mean'),useful_functions_per_1k=('useful_functions_per_1k_est_tokens','mean')).reset_index()
    final_summary.to_csv(RUN_OUT / 'reviewed_test_pathpack.csv',index=False)
    display(final_summary.round(4))


## 8. Аудит, ошибки функций и парные сравнения

Аудит проводится отдельно для каждого split. Он проверяет превышения бюджета, учёт `tokens_est`, дубликаты и потерю выбранных chunks при увеличении бюджета. Вложенность ожидается у `pathpack-v3`; у legacy методов это диагностическое свойство. `diagnostic_cases` показывает «правильный файл, пропущенная функция». Bootstrap сопоставляет одинаковые instance_id.


In [ ]:
audit_reports = {}
for split, path in prediction_paths.items():
    report = audit_predictions([path], SCORING_GOLD, iterations=2000)
    audit_reports[split] = report
    (RUN_OUT / f'audit_{split}.json').write_text(json.dumps(report, ensure_ascii=False, indent=2)+'\n', encoding='utf-8')
    print(split, '| budget violations:', len(report['budget_violations']), '| duplicates:', len(report['duplicate_selection']), '| nested violations:', len(report['nested_violations']))
    display(pd.DataFrame.from_dict(report['groups'], orient='index'))
    display(pd.DataFrame(report['diagnostic_cases']).head(10))
comparisons=[]
for split in RUN_SPLITS:
    for budget in BUDGETS:
        sub=scores[(scores['split']==split) & (scores['budget']==budget)]
        for metric in ('file_recall','function_recall'):
            for candidate in ('pathpack','pathpack-v3'):
                for baseline in ('bm25','graph-expand'):
                    a=sub[sub.method==candidate].dropna(subset=[metric]).set_index('instance_id')[metric].to_dict()
                    b=sub[sub.method==baseline].dropna(subset=[metric]).set_index('instance_id')[metric].to_dict()
                    comparisons.append({'split':split,'budget':budget,'metric':metric,'candidate':candidate,'baseline':baseline,**paired_bootstrap(a,b,iterations=2000)})
ci_table=pd.DataFrame(comparisons)
ci_table.to_csv(RUN_OUT / 'paired_bootstrap.csv',index=False)
display(ci_table)


## 9. Материалы архива v0.3: исторические агрегаты и учебный пример

CSV ниже перенесён из v0.3 и содержит сообщённые пользователем агрегаты прежнего прогона; он не заменяет новые per-issue измерения. Графы учебного примера иллюстрируют поведение алгоритма на синтетическом коде. На четырёх искусственных задачах при бюджете 110 v3 получил 3/4 file hits, legacy PathPack и Graph Expansion — 4/4. Преимущество на реальных ошибках из этого не следует.


In [ ]:
historical_path=ROOT/'reports'/'user_aggregate_2026-11-01.csv'
if historical_path.exists():
    display(pd.read_csv(historical_path))
from IPython.display import Image
for filename in ('user_file_recall.png','user_function_recall.png','v03_selected_call_graph.png'):
    image_path=ROOT/'visualizations'/filename
    if image_path.exists(): display(Image(filename=str(image_path)))
toy_report_path=ROOT/'reports'/'v03_demo'/'audit_demo.json'
if toy_report_path.exists():
    toy_report=json.loads(toy_report_path.read_text())
    display(pd.DataFrame.from_dict(toy_report['groups'],orient='index'))


Новые файлы этого прогона хранятся в `output/benchmark/v03/`. Предыдущие `output/benchmark/predictions.jsonl`, `main_results.csv`, все snapshots и кеши сохранены. Исходный аудит-блокнот v0.3 оставлен как исторический материал; рабочий процесс собран здесь.
